In [1]:
import io
import zipfile
import requests
import frontmatter

In [2]:
def read_repo_data(repo_owner, repo_name):
    """
    Download and parse all markdown files from a GitHub repository.
    
    Args:
        repo_owner: GitHub username or organization
        repo_name: Repository name
    
    Returns:
        List of dictionaries containing file content and metadata
    """
    prefix = 'https://codeload.github.com' 
    url = f'{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main'
    resp = requests.get(url)
    
    if resp.status_code != 200:
        raise Exception(f"Failed to download repository: {resp.status_code}")

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    
    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()

        if not (filename_lower.endswith('.md') 
            or filename_lower.endswith('.mdx')):
            continue
    
        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode('utf-8', errors='ignore')
                post = frontmatter.loads(content)
                data = post.to_dict()
                data['filename'] = filename
                repository_data.append(data)
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue
    
    zf.close()
    return repository_data


In [3]:
zoomcamp = read_repo_data('DataTalksClub', 'data-engineering-zoomcamp')
print(f"Found {len(zoomcamp)} documents")

Found 143 documents


In [4]:
import re

def split_markdown_by_level(text, level=2):
    """
    Split markdown text by a specific header level.
    
    :param text: Markdown text as a string
    :param level: Header level to split on
    :return: List of sections as strings
    """
    # This regex matches markdown headers
    # For level 2, it matches lines starting with "## "
    header_pattern = r'^(#{' + str(level) + r'} )(.+)$'
    pattern = re.compile(header_pattern, re.MULTILINE)

    # Split and keep the headers
    parts = pattern.split(text)
    
    sections = []
    for i in range(1, len(parts), 3):
        # We step by 3 because regex.split() with
        # capturing groups returns:
        # [before_match, group1, group2, after_match, ...]
        # here group1 is "## ", group2 is the header text
        header = parts[i] + parts[i+1]  # "## " + "Title"
        header = header.strip()

        # Get the content after this header
        content = ""
        if i+2 < len(parts):
            content = parts[i+2].strip()

        if content:
            section = f'{header}\n\n{content}'
        else:
            section = header
        sections.append(section)
    
    return sections


In [5]:
zoomcamp_chunks = []

for doc in zoomcamp:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    sections = split_markdown_by_level(doc_content, level=2)
    for section in sections:
        section_doc = doc_copy.copy()
        section_doc['section'] = section
        zoomcamp_chunks.append(section_doc)

In [6]:
print(len(zoomcamp_chunks))
print(zoomcamp_chunks[0].keys())

620
dict_keys(['filename', 'section'])


In [7]:
from minsearch import Index

index = Index(
    text_fields=['section'],  # what fields do you have?
    keyword_fields=[]
)
index.fit(zoomcamp_chunks)

In [8]:
query = 'how do I setup Docker?'
results = index.search(query)
print(f"Number of results: {len(results)}")
print(results[0])

Number of results: 10
{'filename': 'data-engineering-zoomcamp-main/04-analytics-engineering/README.md', 'section': '## Setting up your environment\n\nChoose your setup path:\n\n### 🏠 [Local Setup](setup/local_setup.md)\n\n- **Stack**: DuckDB + dbt Core\n- **Cost**: Free\n- [→ Get Started](setup/local_setup.md)\n\n### ☁️ [Cloud Setup](setup/cloud_setup.md)\n\n- **Stack**: BigQuery + dbt Cloud\n- **Cost**: Free tier available (dbt Cloud Developer), BigQuery costs vary\n- **Requires**: Completed Module 3 with BigQuery data\n- [→ Get Started](setup/cloud_setup.md)'}


In [9]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
from minsearch import VectorSearch
from tqdm.auto import tqdm
import numpy as np

zcamp_embeddings = []

for record in tqdm(zoomcamp_chunks):
    text = record['section']
    v_doc = embedding_model.encode(text)
    zcamp_embeddings.append(v_doc)

zcamp_embeddings = np.array(zcamp_embeddings)

zcamp_vindex = VectorSearch()
zcamp_vindex.fit(zcamp_embeddings, zoomcamp_chunks)

  0%|          | 0/620 [00:00<?, ?it/s]

In [11]:
query = 'how do I setup Docker?'

# text search
text_results = index.search(query)
print("TEXT SEARCH:")
print(text_results[0]['section'][:200])

# vector search
q = embedding_model.encode(query)
vector_results = zcamp_vindex.search(q)
print("\nVECTOR SEARCH:")
print(vector_results[0]['section'])

TEXT SEARCH:
## Setting up your environment

Choose your setup path:

### 🏠 [Local Setup](setup/local_setup.md)

- **Stack**: DuckDB + dbt Core
- **Cost**: Free
- [→ Get Started](setup/local_setup.md)

### ☁️ [Clo

VECTOR SEARCH:
## Basic Docker Commands

Check Docker version:

```bash
docker --version
```

Run a simple container:

```bash
docker run hello-world
```

Run something more complex:

```bash
docker run ubuntu
```

Nothing happens. Need to run it in `-it` mode:

```bash
docker run -it ubuntu
```

We don't have `python` there so let's install it:

```bash
apt update && apt install python3
python3 -V
```


In [12]:
from typing import List, Any

def text_search(query: str, num_results: int = 3):
    return index.search(query, num_results=num_results)

def vector_search(query: str, num_results: int = 3):
    q = embedding_model.encode(query)
    return zcamp_vindex.search(q, num_results=num_results)

def hybrid_search(query: str) -> List[Any]:
    """
    Search the Zoomcamp course sections for relevant information.
    Args:
        query (str): The search query string.
    Returns:
        List[Any]: A list of search results from the course sections.
    """
    text_results = text_search(query, num_results=3)
    vector_results = vector_search(query, num_results=3)
    
    seen_ids = set()
    combined_results = []
    for result in text_results + vector_results:
        key = result['filename'] + result['section']
        if key not in seen_ids:
            seen_ids.add(key)
            # Truncate section to avoid token limits
            r = result.copy()
            r['section'] = result['section'][:500]
            combined_results.append(r)
    
    return combined_results


In [13]:
query = 'how do I setup Docker?'
print(f"Text: {len(text_search(query))} results")
print(f"Vector: {len(vector_search(query))} results")
print(f"Hybrid: {len(hybrid_search(query))} results")

Text: 3 results
Vector: 3 results
Hybrid: 6 results


In [14]:
from groq import Groq
from dotenv import load_dotenv
import os

load_dotenv('../.env')
groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [15]:
from pydantic_ai import Agent
from pydantic_ai.models.groq import GroqModel

model = GroqModel('llama-3.1-8b-instant')

system_prompt = """
You are a helpful assistant for a course. 
Use the search tool to find relevant information before answering questions.
If the first search doesn't give enough information, try different search terms.
"""

agent = Agent(
    model=model,
    system_prompt=system_prompt,
    tools=[hybrid_search]
)

In [16]:
question = "How long does it take to complete the course?"
result = await agent.run(user_prompt=question)
print(result.output)

The information available doesn't specify the exact duration of the course, which could be because the course is self-paced and its duration depends on the student's schedule.


In [17]:
question = "What tools do I need to install for module 1?"
result = await agent.run(user_prompt=question)
print(result.output)

To install the necessary tools for Module 1, you will need to have Docker and PySpark set up on your machine. Docker is used for containerization and infrastructure as code, while PySpark is a high-performance distributed data processing engine that is a key component of the Apache Spark project.

Here are the steps to install Docker and PySpark:

1. Install Docker:
   - Go to the official Docker download page and download the installation package for your operating system.
   - Follow the instructions to install Docker.
   - Once installed, start the Docker service.

2. Install PySpark:
   - You can install PySpark using pip: `pip install pyspark`
   - If you are using a virtual environment, make sure to activate it before installing PySpark.
   - After installing PySpark, you need to download the Spark distribution from the official Apache Spark website and add it to your PATH environment variable.

Additionally, you will need to have Git installed on your machine if you plan to work

In [19]:
from pydantic_ai.messages import ModelMessagesTypeAdapter

def log_entry(agent, messages, source="user"):
    tools = []
    for ts in agent.toolsets:
        tools.extend(ts.tools.keys())

    dict_messages = ModelMessagesTypeAdapter.dump_python(messages)

    system_prompt = agent._instructions
    if not system_prompt:
        try:
            system_prompt = dict_messages[0]['parts'][0]['content']
        except:
            system_prompt = ""

    return {
        "agent_name": agent.name,
        "system_prompt": system_prompt,
        "provider": agent.model.system,
        "model": agent.model.model_name,
        "tools": tools,
        "messages": dict_messages,
        "source": source
    }

In [20]:
import json
import secrets
from pathlib import Path
from datetime import datetime


LOG_DIR = Path('logs')
LOG_DIR.mkdir(exist_ok=True)


def serializer(obj):
    if isinstance(obj, datetime):
        return obj.isoformat()
    raise TypeError(f"Type {type(obj)} not serializable")


def log_interaction_to_file(agent, messages, source='user'):
    entry = log_entry(agent, messages, source)

    ts = entry['messages'][-1]['timestamp']
    # ts_obj = datetime.fromisoformat(ts.replace("Z", "+00:00"))
    ts_str = ts.strftime("%Y%m%d_%H%M%S")
    rand_hex = secrets.token_hex(3)

    filename = f"{agent.name}_{ts_str}_{rand_hex}.json"
    filepath = LOG_DIR / filename

    with filepath.open("w", encoding="utf-8") as f_out:
        json.dump(entry, f_out, indent=2, default=serializer)

    return filepath


In [21]:
question = "How do I set up Docker for the course?"
result = await agent.run(user_prompt=question)
print(result.output)
log_interaction_to_file(agent, result.new_messages())

<function=hybrid_search>{"query": "setting up Docker for Zoomcamp course"}


PosixPath('logs/agent_20260329_185130_64f41c.json')

In [22]:
import time

questions = [
    "What tools do I need to install for module 1?",
    "How do I set up BigQuery?",
    "What is the homework for the Kafka module?",
    "How do I connect Spark to GCS?",
    "What are the prerequisites for the course?"
]

for q in tqdm(questions):
    print(q)
    result = await agent.run(user_prompt=q)
    print(result.output)
    log_interaction_to_file(agent, result.new_messages())
    print()
    time.sleep(30)

  0%|          | 0/5 [00:00<?, ?it/s]

What tools do I need to install for module 1?
Based on the search results, the tools required for Module 1 are Docker and PySpark. Docker is mentioned in the homework instructions for Module 6, but it's also relevant for Module 1. PySpark is mentioned in the homework instructions for Module 5, but it's also relevant for Module 1.

To confirm this, you can try searching for "Module 1 required tools" or "Module 1 tools download" again, or look for more specific information in the course materials.

How do I set up BigQuery?
To set up BigQuery, you should follow these steps:

1. Verify your BigQuery setup, including checking your service account and making sure it has the required permissions (BigQuery Data Editor, BigQuery Job User, and BigQuery User).
2. Create a new service account or download a new key file if needed.
3. Choose the recommended setup path, which is using BigQuery with dbt Cloud. This is the fastest way to get started and closest to how teams actually use dbt in product

In [26]:
import random 

sample = random.sample(zoomcamp_chunks, 10)
prompt_docs = [d['section'] for d in sample] 

In [27]:
evaluation_prompt = """
Use this checklist to evaluate the quality of an AI agent's answer (<ANSWER>) to a user question (<QUESTION>).
We also include the entire log (<LOG>) for analysis.

For each item, check if the condition is met. 

Checklist:

- instructions_follow: The agent followed the user's instructions (in <INSTRUCTIONS>)
- instructions_avoid: The agent avoided doing things it was told not to do  
- answer_relevant: The response directly addresses the user's question  
- answer_clear: The answer is clear and correct  
- answer_citations: The response includes proper citations or sources when required  
- completeness: The response is complete and covers all key aspects of the request
- tool_call_search: Is the search tool invoked? 

Output true/false for each check and provide a short explanation for your judgment.
""".strip()

In [28]:
from pydantic import BaseModel

class EvaluationCheck(BaseModel):
    check_name: str
    justification: str
    check_pass: bool

class EvaluationChecklist(BaseModel):
    checklist: list[EvaluationCheck]
    summary: str


In [29]:
eval_agent = Agent(
    name='eval_agent',
    model= GroqModel('llama-3.3-70b-versatile'),
    instructions=evaluation_prompt,
    output_type=EvaluationChecklist
)

In [30]:
system_prompt = """
You are a helpful assistant for the Data Engineering Zoomcamp course.
Use the search tool to find relevant information before answering questions.
If the first search doesn't give enough information, try different search terms.

Always include references by citing the filename of the source material you used.
When citing the reference, replace "data-engineering-zoomcamp-main" with the full path to the GitHub repository: "https://github.com/DataTalksClub/data-engineering-zoomcamp/blob/main/"
Format: [LINK TITLE](FULL_GITHUB_LINK)

If the search doesn't return relevant results, let the user know and provide general guidance.
""".strip()

agent = Agent(
    name="zoomcamp_agent",
    model=model,
    system_prompt=system_prompt,
    tools=[hybrid_search]
)

In [31]:
result = await agent.run(user_prompt="How do I set up Docker?")
print(result.output)

Based on the search results, to set up Docker, you can start by running a simple container using the command `docker run hello-world` followed by `docker run ubuntu` to run a more complex container. If you want to interact with the container, use the `-it` flag.

To get information on Docker, use the command `docker --help`. For specific Docker commands like `docker build` and `docker run`, use the `-help` flag.

Additionally, the search results provide examples of setting up Docker containers for specific use cases, such as running Postgres with pgAdmin.

[Docker Commands in the Data Engineering Zoomcamp]( https://github.com/DataTalksClub/data-engineering-zoomcamp/blob/main/01-docker-terraform/docker-sql/01-introduction.md)
